In [17]:
import numpy as np
import time, os, sys
import shutil
#from tifffile import imwrite, imsave
from skimage import img_as_float32
#from urllib.parse import urlparse
import matplotlib.pyplot as plt
import matplotlib as mpl
import skimage
from sklearn.model_selection import train_test_split
%matplotlib inline
mpl.rcParams['figure.dpi'] = 300
from cellpose import core, utils, io, models, metrics, train, transforms

use_GPU = core.use_gpu()
yn = ['NO', 'YES']
print(f'>>> GPU activated? {yn[use_GPU]}')

core.assign_device(use_torch=True, gpu=True, device=0)

import torch


>>> GPU activated? YES


In [ ]:
training_path = "D:/Projects/Mikala/images/training_norm/TJ/xy/"
model_path = "D:/Projects/Mikala/images/training_norm/models" + "/models/"

modelName = 'TJ_xy_pipeline_norm_Mikala'

#images = io.get_image_files(training_path, '_seg.npy', imf=None, look_one_level_down=False)
#print(len(images))
#print(images)

#dataset = io.load_images_labels(training_path, mask_filter='_seg.npy', image_filter=None, look_one_level_down=False)

#dataset = io.load_train_test_data(training_path, test_dir=None, image_filter=None, mask_filter='_seg.npy', look_one_level_down=False)
#print(dataset)
# list to store files
annotation_files = []
# Iterate directory
for file in os.listdir(training_path):
    # check only text files
    if file.endswith('.npy'):
        annotation_files.append(file)
#print(annotation_files)

image_files = [os.path.join(training_path,sub.replace('_seg.npy', '.tif')) for sub in annotation_files]
annotation_files = [os.path.join(training_path,sub) for sub in annotation_files]
print(image_files)
print(annotation_files)

images_train, images_test, masks_train, masks_test = train_test_split(image_files, annotation_files, test_size=0.1, random_state=42)
print(images_train)
print(masks_train)
print(images_test)
print(masks_test)

## Train TJ dataset

In [ ]:
model = models.CellposeModel(model_type="nuclei", gpu=True)
io.logger_setup()

normalize_custom_training = {
    "lowhigh": [0,255],
    "percentile": None,
    "normalize": True,
    "norm3D": False, 
    "sharpen_radius": 0,
    "smooth_radius": 0,
    "tile_norm_blocksize": 0,
    "tile_norm_smooth3D": 1,
    "invert": False
}

#output = io.load_train_test_data(train_dir, test_dir, image_filter="_img",
#                                mask_filter="_masks", look_one_level_down=False)
#images, labels, image_names, test_images, test_labels, image_names_test = output

model_path = train.train_seg(model.net, 
                            train_files=images_train, train_labels_files=masks_train,
                            normalize=normalize_custom_training, channels=[1,0], compute_flows=False,
                            test_files=images_test, test_labels_files=masks_test,
                            weight_decay=1e-5, SGD=True, learning_rate=0.005,
                            n_epochs=2000,  
                            save_path= model_path, model_name=modelName)




# cellpose.train.train_seg(net, train_data=None, train_labels=None, train_files=None, train_labels_files=None, train_probs=None, test_data=None, test_labels=None,
#                           test_files=None, test_labels_files=None, test_probs=None, load_files=True, batch_size=8, learning_rate=0.005, n_epochs=2000,
#                          weight_decay=1e-05, momentum=0.9, SGD=False, channels=None, channel_axis=None, rgb=False, normalize=True, compute_flows=False,
#                            save_path=None, save_every=100, nimg_per_epoch=None, nimg_test_per_epoch=None, rescale=True, scale_range=None, 
#                          bsize=224, min_train_masks=5, model_name=None)



## Set paths for analysis

In [18]:
dataset_path = 'D:\\Mikala\\images\\GT\\raw'
output_path= 'D:\\Mikala\\images\\tunning\\cp4_3D_300_smooth2'

## Ev TJ data

## VASA Ev


In [ ]:
#python -m cellpose --dir D:\Projects\Mikala\images\downsampled\02072024\ch4 --pretrained_model Mikala_bright_nuclei --chan 1 --chan2 2 --save_tif --do_3D --use_gpu --verbose
Channels = [0, 0] 

Det_threshold = 0
Flow_threshold=0.4
Cellprob_threshold=0 
Batch_size = 64
Do_3D=True 
Anisotropy = 1.78
Flow3D_smooth= 2 
Stitch_threshold=0.1
Min_size=15
Z_axis = 0


images_path = os.path.join(dataset_path,'ch1')
VASA_output_path = os.path.join(output_path, 'VASA_ch1')


model_path = "D:\\Mikala\\images\\training\\VASA\\VASA_core_cp4" +"\\models"
modelName = 'VASA_CORE_cp4_xy_07312025_300epoch'






normalize_custom = {
    "lowhigh": None ,
    "percentile": [1.0,99.0],
    "normalize": True,
    "norm3D": True,
    "sharpen_radius": 0,
    "smooth_radius": 0,
    "tile_norm_blocksize": 0,
    "tile_norm_smooth3D": 1,
    "invert": False
}


print('Processing VASA channel:')

# model_type='cyto' or model_type='nuclei'
fullpath_model = os.path.join(model_path,modelName)
print('Cellpose model path = ' + fullpath_model)

model = models.CellposeModel(gpu=True, pretrained_model=fullpath_model)

#model = models.CellposeModel(pretrained_model=os.path.join(model_path,modelName), gpu = True, diam_mean=diameter)

isExist = os.path.exists(VASA_output_path)
if not isExist:

   # Create a new directory because it does not exist
   os.makedirs(VASA_output_path)


target_files = []
# Iterate directory
for file in os.listdir(images_path):
    # check only text files
    if file.endswith('downs.tiff'):
        target_files.append(file)

#shutil.rmtree(output_path)


for f in target_files:
    shutil.copy(os.path.join(images_path,f), os.path.join(VASA_output_path,f))

#target_files = [os.path.join(output_path,sub) for sub in target_files]
#img = skimage.io.imread('/path/to/image/')

# channel to segment and nuclear channel 
# numbering starts at 1 
# for your single channel image use [0, 0] 
# for the multi channel image it's [3, 0]



for idx, image in enumerate(target_files):
    print('Processing file ' + str(idx+1) + ' of ' + str(len(target_files)))
    imPath = os.path.join(VASA_output_path,image)
    print("Performing prediction on: "+ imPath)   
    
    stack = io.imread(imPath)
    short_name = os.path.splitext(image)
    n_plane = stack.shape[0]
    #prediction_stack = np.zeros((n_plane, stack.shape[1], stack.shape[2]))


 
    masks, flows, styles = model.eval(stack, batch_size = Batch_size, 
                                      normalize = normalize_custom, stitch_threshold=Stitch_threshold,
                                      do_3D=Do_3D, anisotropy = Anisotropy, flow3D_smooth = Flow3D_smooth, min_size=Min_size, 
                                      cellprob_threshold=Cellprob_threshold,z_axis = Z_axis)
    
  
    prediction_stack_32 = img_as_float32(masks, force_copy=False)

    print('Saving output files to directory:' + VASA_output_path)
    os.chdir(VASA_output_path)
   
    io.masks_flows_to_seg(stack, masks, flows, str(short_name[0]), channels=Channels)
    io.save_masks(stack, masks, flows, str(short_name[0]), png=False, tif=True, channels=Channels)

        

    

Processing VASA channel:
Cellpose model path = D:\Mikala\images\training\VASA\VASA_core_cp4\models\VASA_CORE_cp4_xy_07312025_300epoch
Processing file 1 of 3
Performing prediction on: D:\Mikala\images\tunning\cp4_3D_300_smooth2\VASA_ch1\02222025_control_dapi_488_555_647_ovary1_ch1_downs.tiff


100%|██████████| 230/230 [00:00<00:00, 685.49it/s]


Saving output files to directory:D:\Mikala\images\tunning\cp4_3D_300_smooth2\VASA_ch1
Processing file 2 of 3
Performing prediction on: D:\Mikala\images\tunning\cp4_3D_300_smooth2\VASA_ch1\02222025_cora_dapi_488_555_647_ovary1_ch1_downs.tiff


100%|██████████| 227/227 [00:00<00:00, 899.79it/s]


Saving output files to directory:D:\Mikala\images\tunning\cp4_3D_300_smooth2\VASA_ch1
Processing file 3 of 3
Performing prediction on: D:\Mikala\images\tunning\cp4_3D_300_smooth2\VASA_ch1\02262025_atp_dapi_488_555_647_ovary1_ch1_downs.tiff


100%|██████████| 237/237 [00:00<00:00, 986.48it/s]


Saving output files to directory:D:\Mikala\images\tunning\cp4_3D_300_smooth2\VASA_ch1
